# Build 4 — the basin value head

Residual stream at token t → distribution over basins. Trained on the basin-pipeline
artifacts from `13_basin_datagen.ipynb` (labels = endpoint cluster ∘ metastable merge;
features = teacher-forced residuals from the stored token ids — nothing is regenerated).

1. **Extract** — one forward pass per rollout, harvest residuals at layers 9/18/27 and
   logprob/entropy baseline features at depths 8..56 (per organism, GPU)
2. **Train** — linear softmax head per layer vs the logprob-only baseline; prompt-level
   splits; soft-target eval at transition fork points (empirical distribution of the 8
   resampled futures)
3. **Ribbon** — P(basin) at every token of showcase rollouts: watch the thought commit

Needs the `data/basin_corpus_xl` artifacts in this runtime: either run notebook 13 first,
or upload its `basin_xl_results.zip` to `/content` and the cell below unpacks it. A100
recommended (extraction is 3 model loads; training is minutes).

In [ ]:
import os
if not os.path.exists("/content/dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git /content/dt_rl
%cd /content/dt_rl
!git pull
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print("no HF_TOKEN from userdata:", e)
%pip install -q -U transformers accelerate scikit-learn

In [ ]:
import pathlib
if not pathlib.Path("data/basin_corpus_xl/basins/basins.json").exists():
    if pathlib.Path("/content/basin_xl_results.zip").exists():
        !unzip -qo /content/basin_xl_results.zip -d /content/dt_rl
    else:
        raise SystemExit("no basin artifacts: run 13_basin_datagen.ipynb in this "
                         "runtime first, or upload basin_xl_results.zip to /content")
!ls data/basin_corpus_xl

## 1 — extract residuals + baseline features (per organism)

~8.4k rollouts × 11 depths per organism; the fork depths 12/24/36/48 are in the grid so
the soft-target join works. ~1 GB of fp16 features per organism.

In [ ]:
for org in ["base", "dark", "clinical-depression"]:
    !python scripts/basin_value_extract.py --organism {org} --corpus data/basin_corpus_xl
!ls -lh data/basin_corpus_xl/value_feats

## 2 — train the head: layer sweep vs the logprob-only baseline

The claim the head must earn: beat the logprob baseline by a clear margin, especially at
small t — that gap is trajectory information the token distribution doesn't carry.

In [ ]:
!python scripts/basin_value_train.py --corpus data/basin_corpus_xl
from IPython.display import Markdown, Image, display
display(Image("data/basin_corpus_xl/value_head/accuracy_vs_t.png"))
display(Markdown(open("data/basin_corpus_xl/value_head/report.md").read()))

## 3 — ribbon plots: P(basin) along the trajectory

Test-split rollouts, spread across basins. The dashed line is where max P(basin)
first clears 0.9 — the head's read of the commitment token.

In [ ]:
for org in ["base", "dark", "clinical-depression"]:
    !python scripts/basin_value_ribbon.py --organism {org} --corpus data/basin_corpus_xl
from IPython.display import Image, display
for org in ["base", "dark", "clinical-depression"]:
    display(Image(f"data/basin_corpus_xl/value_head/ribbon_{org}.png"))

## 4 — save

`value_head.npz` (+ ribbons/report) is small; the fp16 feature files stay in the runtime
unless you want them for later heads (MLP, per-organism, calibration).

In [ ]:
!zip -qr value_head_results.zip data/basin_corpus_xl/value_head
!ls -lh value_head_results.zip
# from google.colab import files; files.download("value_head_results.zip")
# feature files too (large): !zip -qr value_feats.zip data/basin_corpus_xl/value_feats